# EDA Overview

Quick exploratory analysis to understand:
- Revenue by country
- Orders by month
- Customer distribution

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../data/clean_retail.csv")
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

# standard helper columns
df["Month"] = df["InvoiceDate"].dt.to_period("M").dt.to_timestamp()

# order-level table (one row per invoice)
orders = (
    df.groupby("Invoice", as_index=False)
      .agg(
          OrderValue=("Revenue", "sum"),
          OrderDate=("InvoiceDate", "min"),
          CustomerID=("Customer ID", "first"),
          Country=("Country", "first")
      )
)
orders["OrderMonth"] = orders["OrderDate"].dt.to_period("M").dt.to_timestamp()

df.head()

## Revenue by Country

In [ ]:
country_revenue = (
    df.groupby("Country", as_index=False)["Revenue"]
      .sum()
      .sort_values("Revenue", ascending=False)
)

country_revenue.head(15)

In [ ]:
top_c = country_revenue.head(10).copy()

plt.figure(figsize=(10,5))
plt.barh(top_c["Country"], top_c["Revenue"])
plt.gca().invert_yaxis()
plt.title("Top 10 Countries by Revenue")
plt.xlabel("Revenue")
plt.ylabel("Country")
plt.tight_layout()
plt.show()

## Orders by Month

In [ ]:
monthly_orders = (
    orders.groupby("OrderMonth", as_index=False)["Invoice"]
          .nunique()
          .rename(columns={"Invoice": "Orders"})
          .sort_values("OrderMonth")
)

monthly_revenue = (
    df.groupby("Month", as_index=False)["Revenue"]
      .sum()
      .rename(columns={"Month": "OrderMonth"})
      .sort_values("OrderMonth")
)

monthly_orders.head(), monthly_revenue.head()

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(monthly_orders["OrderMonth"], monthly_orders["Orders"])
plt.title("Monthly Orders Trend")
plt.xlabel("Month")
plt.ylabel("Orders")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(monthly_revenue["OrderMonth"], monthly_revenue["Revenue"])
plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.tight_layout()
plt.show()

## Customer Distribution


In [ ]:
customer = (
    orders.groupby("CustomerID", as_index=False)
          .agg(
              TotalSpend=("OrderValue", "sum"),
              Orders=("Invoice", "nunique"),
              FirstOrder=("OrderDate", "min"),
              LastOrder=("OrderDate", "max")
          )
)

customer["AvgOrderValue"] = customer["TotalSpend"] / customer["Orders"]
customer.head()

In [ ]:
plt.figure(figsize=(8,4))
plt.hist(np.log1p(customer["TotalSpend"]), bins=50)
plt.title("Customer Total Spend Distribution (log scale)")
plt.xlabel("log(1 + TotalSpend)")
plt.ylabel("Customers")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,4))
plt.hist(customer["Orders"], bins=50)
plt.title("Orders per Customer Distribution")
plt.xlabel("Number of Orders")
plt.ylabel("Customers")
plt.tight_layout()
plt.show()

In [ ]:
customer_sorted = customer.sort_values("TotalSpend", ascending=False).copy()
customer_sorted["CumRevenueShare"] = customer_sorted["TotalSpend"].cumsum() / customer_sorted["TotalSpend"].sum()

top_10pct_n = int(np.ceil(0.10 * len(customer_sorted)))
top_10pct_share = customer_sorted.iloc[top_10pct_n-1]["CumRevenueShare"]

top_10pct_n, top_10pct_share